&nbsp;

# Prepare Gene Ontology - gene (transcript) Association File

- `GOAtools` で使用するための association file を作成する
- Experimental evidence codes のみでエンリッチメント解析をしてみるとどうなるのか調べたいと考え、実行

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

## QuickGO annotation (human all)

- `NOT`を含むアノテーションは、今回考慮しない
- + [GO evidence code](https://geneontology.org/docs/guide-go-evidence-codes/) に基づき、「Experimental evidence codes」のみを選んで association file を作成してみることに

In [3]:
quickgo_human_all = pl.read_csv(
    "../../Data/Database/QuickGO/QuickGO-annotations-human-all-1792593337498-20261021.tsv.gz",
    # QuickGO export is tab-separated. Default comma split treats the header as one field.
    separator="\t"
).filter(
    # "|" is literal. As a regex, "NOT|" matches every string.
    ~(pl.col("QUALIFIER").str.contains("NOT|", literal=True)) # 今回はこの NOT を含むアノテーションを考慮しない
).filter(
    (
        # Experimental evidence codes
        # https://geneontology.org/docs/guide-go-evidence-codes/
        (pl.col("GO EVIDENCE CODE") == "EXP")
        | (pl.col("GO EVIDENCE CODE") == "IDA")
        | (pl.col("GO EVIDENCE CODE") == "IPI")
        | (pl.col("GO EVIDENCE CODE") == "IMP")
        | (pl.col("GO EVIDENCE CODE") == "IGI")
        | (pl.col("GO EVIDENCE CODE") == "IEP")
        | (pl.col("GO EVIDENCE CODE") == "HTP")
        | (pl.col("GO EVIDENCE CODE") == "HDA")
        | (pl.col("GO EVIDENCE CODE") == "HMP")
        | (pl.col("GO EVIDENCE CODE") == "HGI")
        | (pl.col("GO EVIDENCE CODE") == "HEP")
    )
    
)

print(quickgo_human_all.select("GENE PRODUCT DB").unique())

pl.Config.set_tbl_rows(22)
print(quickgo_human_all.select("QUALIFIER").sort("QUALIFIER").unique())

pl.Config.set_tbl_rows(2)
display(quickgo_human_all)

shape: (2, 1)
┌─────────────────┐
│ GENE PRODUCT DB │
│ ---             │
│ str             │
╞═════════════════╡
│ UniProtKB       │
│ RNAcentral      │
└─────────────────┘
shape: (13, 1)
┌─────────────────────────────────┐
│ QUALIFIER                       │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ acts_upstream_of                │
│ acts_upstream_of_negative_effe… │
│ acts_upstream_of_or_within      │
│ acts_upstream_of_or_within_neg… │
│ acts_upstream_of_or_within_pos… │
│ acts_upstream_of_positive_effe… │
│ colocalizes_with                │
│ contributes_to                  │
│ enables                         │
│ involved_in                     │
│ is_active_in                    │
│ located_in                      │
│ part_of                         │
└─────────────────────────────────┘


GENE PRODUCT DB,GENE PRODUCT ID,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""UniProtKB""","""A0A023T787""","""RBM8""","""located_in""","""GO:0016607""","""nuclear speck""","""ECO:0000314""","""IDA""","""GO_REF:0000052""",null,9606,"""HPA""",null,20251115,"""RNA-binding protein 8A""","""RBM8,Y14""","""protein""",null,"""C"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RNAcentral""","""URS0000EEF6F1_9606""","""Homo sapiens (human) HSALNT039…","""located_in""","""GO:0005634""","""nucleus""","""ECO:0000314""","""IDA""","""PMID:32969152""",null,9606,"""MGI""",null,20230912,"""Homo sapiens (human) HSALNT039…",null,"""RNA""",null,"""C"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (2) Transcript - GO association file

- この association file は、必ずしも、背景の transcript と数をあわせる必要はなく、あくまで包摂していることが重要

In [4]:
all_transcript = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode_v49_tx2gene_chr_patch_hapl_scaff_v260909.tsv",
    separator="\t"
)

display(all_transcript)

transcript_id,gene_id
str,str
"""ENST00000832824.1""","""ENSG00000290825.2"""
…,…
"""ENST00000839853.1""","""ENSG00000309258.1"""


&nbsp;

&nbsp;

## TogoID process (2026-10-22)

In [ ]:
# # transcript list
# all_quantified_transcript_list = (
#     all_transcript
#     .select(pl.col("transcript_id"))
#     .unique()
# ).write_csv(
#     "../../out/enrichment_analysis/togoid/all_transcript_list.txt",
#     separator="\t",
#     include_header=False
# )

&nbsp;

&nbsp;

## Ensembl Transcript ID - Ensembl Protein ID - UniProt ID

- TogoID access: 2026-10-22
- `Ensembl transcript - is translated to - Ensembl protein - is equivalent to - UniProt`
- command: `bash ./scripts/togoid_convert.sh ./out/enrichment_analysis/togoid/all_transcript_list.txt ensembl_transcript,ensembl_protein,uniprot ./out/enrichment_analysis/togoid/all_transcript_uniprot.tsv`

In [5]:
uniprot_convert = pl.read_csv(
    "../../out/enrichment_analysis/togoid/all_transcript_uniprot.tsv.gz",
    separator="\t"
).rename(
    {
        "uniprot_id": "GENE PRODUCT ID"
    }
)

display(uniprot_convert)

ensembl_transcript_id,ensembl_protein_id,GENE PRODUCT ID
str,str,str
"""ENST00000886743""","""ENSP00000556802""","""A0ACI8SMA2"""
…,…,…
"""ENST00000445230""",null,null


In [6]:
uniprot_convert_filtered = uniprot_convert.filter(
    pl.col("GENE PRODUCT ID").is_not_null()
)

display(uniprot_convert_filtered)

ensembl_transcript_id,ensembl_protein_id,GENE PRODUCT ID
str,str,str
"""ENST00000886743""","""ENSP00000556802""","""A0ACI8SMA2"""
…,…,…
"""ENST00000908583""","""ENSP00000578642""","""Q13421"""


&nbsp;

&nbsp;

## Protein coding transcript - GO association

- まずは、UniProt ID をキーとして、対応付ける
- また、UniProtKB 由来の association file を作成し、これをバックグラウンドにする

In [7]:
protein_coding_transcript_go = uniprot_convert_filtered.join(
    quickgo_human_all,
    on="GENE PRODUCT ID",
    how="inner"
).drop(
    "ensembl_protein_id"
).rename(
    {
        "ensembl_transcript_id": "transcript_id"
    }
)

display(protein_coding_transcript_go)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000612959""","""A0A024QZ33""","""UniProtKB""","""NSRP1""","""located_in""","""GO:0005654""","""nucleoplasm""","""ECO:0000314""","""IDA""","""GO_REF:0000052""",null,9606,"""HPA""",null,20251115,"""Nuclear speckle splicing regul…","""NSRP1""","""protein""",null,"""C"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000610140""","""X6RM59""","""UniProtKB""","""NT5C3A""","""located_in""","""GO:0005783""","""endoplasmic reticulum""","""ECO:0000314""","""IDA""","""GO_REF:0000052""",null,9606,"""HPA""",null,20251115,"""5'-nucleotidase""","""NT5C3A""","""protein""",null,"""C"""


&nbsp;

&nbsp;

&nbsp;

## Non-coding Transcript - GO association

- Transcript ID のバージョンを落とす
- RNACentral ID に 「_9606」をつける

In [8]:
transcript_rnacentral_id = pl.read_csv(
    "../../Data/Database/RNACentral/release27/database_mapping/ensembl_gencode.tsv",
    separator="\t",
    has_header=False,
    new_columns=[
        "RNACentral_ID",
        "relationship",
        "transcript_id",
        "taxon_id",
        "type",
        "gene_id"
    ]
).filter(
    pl.col("taxon_id") == 9606
).select(
    pl.col("transcript_id"),
    pl.col("RNACentral_ID").alias("GENE PRODUCT ID")
).with_columns(
    # Append the human NCBI taxon id used by RNAcentral, e.g. URS0002A62C26_9606
    pl.concat_str(pl.col("GENE PRODUCT ID"), pl.lit("_9606")).alias("GENE PRODUCT ID"),
)

display(transcript_rnacentral_id)

transcript_id,GENE PRODUCT ID
str,str
"""ENST00000585414""","""URS0000000055_9606"""
…,…
"""ENST00000787161""","""URS000344EFF2_9606"""


In [9]:
non_coding_transcript_go = transcript_rnacentral_id.join(
    quickgo_human_all,
    on="GENE PRODUCT ID",
    how="inner"
)

display(non_coding_transcript_go)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000501122""","""URS0000204428_9606""","""RNAcentral""","""Homo sapiens (human) non-prote…","""acts_upstream_of""","""GO:1901647""","""positive regulation of synovio…","""ECO:0000315""","""IMP""","""PMID:28379604""",null,9606,"""ARUK-UCL""",null,20230522,"""Homo sapiens (human) non-prote…",null,"""RNA""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000651587""","""URS0000E8234C_9606""","""RNAcentral""","""Homo sapiens (human) Pvt1 onco…","""involved_in""","""GO:2000627""","""positive regulation of miRNA c…","""ECO:0000314""","""IDA""","""PMID:28258379""",null,9606,"""BHF-UCL""","""has_input(RNAcentral:URS00006A…",20230524,"""Homo sapiens (human) Pvt1 onco…",null,"""RNA""",null,"""P"""


&nbsp;

&nbsp;

&nbsp;

## (IMPORTANT!) 整合性 check

- [URS0002A5FA5F_9606](https://rnacentral.org/rna/URS0002A5FA5F/9606) みたいに、QuickGOを見ると Rfam にアノテーションされているが、RNACentral を見ると、アノテーションがない、という齟齬が見られることから、両方のソースをみて check する
- RNACentral の release より QuickGO が遅れてアノテーションされている？

In [10]:
rnacentral_annotation = pl.read_csv(
    "../../Data/Database/RNACentral/release27/rnacentral_rfam_annotations.tsv.gz",
    separator="\t",
    has_header=False,
    new_columns=[
       "RNACentral_ID",
        "GO TERM",
        "WITH/FROM"
    ],
    n_threads=16
).filter(
    # only human
    pl.col("RNACentral_ID").str.contains("_9606")
).rename(
    {
        "RNACentral_ID": "GENE PRODUCT ID"
    }
)

display(rnacentral_annotation)

GENE PRODUCT ID,GO TERM,WITH/FROM
str,str,str
"""URS0000001346_9606""","""GO:0030533""","""Rfam:RF00005"""
…,…,…
"""URS0003707711_9606""","""GO:0006396""","""Rfam:RF00277"""


&nbsp;

&nbsp;

### RNACentral 以外の アノテーションは残しておく

In [11]:
# Keep non-RNAcentral assignments as they are.
# Only RNAcentral rows are checked against the release-27 Rfam table.
non_coding_transcript_go_rnacentral = non_coding_transcript_go.filter(
    pl.col("ASSIGNED BY") == "RNAcentral"
)
non_coding_transcript_go_other = non_coding_transcript_go.filter(
    pl.col("ASSIGNED BY") != "RNAcentral"
)

non_coding_transcript_go_rnacentral_checked = non_coding_transcript_go_rnacentral.join(
    rnacentral_annotation,
    on=[
        "GENE PRODUCT ID",
        "GO TERM",
        "WITH/FROM"
    ],
    how="inner"
)

non_coding_transcript_go_rnacentral_validated = pl.concat(
    [
        non_coding_transcript_go_rnacentral_checked,
        non_coding_transcript_go_other
    ],
    how="vertical_relaxed"
).sort(
    "transcript_id",
    "GO TERM"
)

display(non_coding_transcript_go_rnacentral_validated)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000501122""","""URS0000204428_9606""","""RNAcentral""","""Homo sapiens (human) non-prote…","""involved_in""","""GO:0000512""","""lncRNA-mediated post-transcrip…","""ECO:0000314""","""IDA""","""PMID:28379604""",null,9606,"""ARUK-UCL""","""has_input(RNAcentral:URS000039…",20230522,"""Homo sapiens (human) non-prote…",null,"""RNA""",null,"""P"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000651587""","""URS0000E8234C_9606""","""RNAcentral""","""Homo sapiens (human) Pvt1 onco…","""involved_in""","""GO:2000627""","""positive regulation of miRNA c…","""ECO:0000314""","""IDA""","""PMID:28258379""",null,9606,"""BHF-UCL""","""has_input(RNAcentral:URS00006A…",20230524,"""Homo sapiens (human) Pvt1 onco…",null,"""RNA""",null,"""P"""


&nbsp;

&nbsp;

&nbsp;

## Concat Process

In [12]:
all_go_association = pl.concat(
    [
        protein_coding_transcript_go,
        non_coding_transcript_go_rnacentral_validated
    ],
    how="vertical_relaxed"
)

display(all_go_association)

transcript_id,GENE PRODUCT ID,GENE PRODUCT DB,SYMBOL,QUALIFIER,GO TERM,GO NAME,ECO ID,GO EVIDENCE CODE,REFERENCE,WITH/FROM,TAXON ID,ASSIGNED BY,ANNOTATION EXTENSION,DATE,GENE_PRODUCT_NAME,GENE_PRODUCT_SYNONYMS,GENE_PRODUCT_TYPE,INTERACTING TAXON ID,GO ASPECT
str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str
"""ENST00000612959""","""A0A024QZ33""","""UniProtKB""","""NSRP1""","""located_in""","""GO:0005654""","""nucleoplasm""","""ECO:0000314""","""IDA""","""GO_REF:0000052""",null,9606,"""HPA""",null,20251115,"""Nuclear speckle splicing regul…","""NSRP1""","""protein""",null,"""C"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000651587""","""URS0000E8234C_9606""","""RNAcentral""","""Homo sapiens (human) Pvt1 onco…","""involved_in""","""GO:2000627""","""positive regulation of miRNA c…","""ECO:0000314""","""IDA""","""PMID:28258379""",null,9606,"""BHF-UCL""","""has_input(RNAcentral:URS00006A…",20230524,"""Homo sapiens (human) Pvt1 onco…",null,"""RNA""",null,"""P"""


In [13]:
all_go_association.write_csv(
    "../../out/enrichment_analysis/transcript_go_association/all_transcript_go_relation_quickgo_only_exp_evidence.tsv.gz",
    separator="\t",
    compression="gzip",
    compression_level=9
)

&nbsp;

&nbsp;

## (2) Create Transcript - GO association File

In [14]:
transcript_go_association = all_go_association.select(
    "transcript_id",
    "GO TERM"
).group_by(
    "transcript_id"
).agg(
    # unique() is list[str] only after agg finishes.
    pl.col("GO TERM").unique().alias("GO term accessions")
).with_columns(
    # Join whole GO ids. map_elements would join the characters of each id.
    pl.col("GO term accessions").list.sort().list.join(";")
).sort(
    "transcript_id"
)

display(transcript_go_association)

transcript_id,GO term accessions
str,str
"""ENST00000000233""","""GO:0005515;GO:0005765;GO:00057…"
…,…
"""ENST00000973046""","""GO:0003723;GO:0005515;GO:00056…"


In [ ]:
# transcript_go_association.write_csv(
#     "../../out/enrichment_analysis/transcript_go_association/transcript_go_association_exp_evidence.txt",
#     separator="\t",
#     include_header=False
# )